## Newton's Method - Introduction

This notebook walks through the optimization method employed by the normal distribution transform.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import utils.plotting as plot
import utils.mat_ops as mat
import MeshSampler as ms
import plotly.graph_objects as go

from tqdm import tqdm
from copy import copy
from typing import Callable, cast
from IPython.display import clear_output

#### Newton's method in 1-D

To find the minimum of the function, $f(x)$, we would simply compute $f'(x) = 0$. If $x$ cannot be isolated in $f'(x)$, however, solving for the minima requires an iterative method. An example is a curve with the equation:

\begin{align*}
f(x) = x^2 + e^{-x}
\end{align*}

whose derivative is

\begin{align*}
f'(x) = 2x - e^{-x}
\end{align*}

These functions are defined and plotted below.

In [ ]:
def f( x: float ) -> float:
    return x ** 2 + np.exp( -x )

def f_d( x: float ) -> float:
    return 2 * x - np.exp( -x )

plot.plot_univariate_functions( funcs = [f, f_d], labels = ['f(x)', "f'(x)"], x_range = (-1, 1) ).show()

To first show gradient descent in action, we'll start with a derivate for which the local minima is easily found, $f(x) = ( x - 2 )^2$. The minima can be found by setting the derivative equal to $0$, $f'(x) = 2 ( x - 2 ) = 2x - 4 = 0 \rightarrow x = 2$. This can be confirmed visually in the plot below.

In [ ]:
def f( x: float ) -> float:
    return ( x - 2 ) ** 2

plot.plot_univariate_functions( funcs = [f], labels = ['f(x)'], x_range = (-10, 10) ).show()

If it were not so easy to determine the value of $x$ where $f'(x) = 0$, we could use Newton's method to iteratively find the minima. Let's say we had an initial guess that $f'(-2) = 0$. Our initial guess, $x_n = -2$, could then be improved by computing $x_{n + 1} = x_n - \alpha d_k$, where $\alpha$ is the learning rate (or step size) in the direction of $d_k$, which is the direction of descent toward the minima. Since the derivative $f'(x)$ represents the tangent line of $f(x)$ at any given $x$, we can simply set $d_k = f'(x)$. Thus, the iterative equation is then $x_{n + 1} = x_n - \alpha f'(x_n)$. Below are examples solving the iterative method with a termination criteria of $\epsilon < 0.001$, meaning that the algorithm stops when $| \Delta f(x) | = | f(x_{n + 1}) - f(x_n) | < 0.001$.

In [ ]:
def gradient_descent_1d_func_gen( f: Callable[[float], float], f_d: Callable[[float], float], x_0: float, learning_rate: float = 0.2, epsilon: float = 0.001 ) -> list[Callable[[float], float]]:
    
    delta_f: float = epsilon * 10
    x: float = x_0

    f_n: list[Callable[[float], float]] = [f]
    
    while( delta_f > epsilon ):
        m = f_d( x )
        b = f( x ) - m * x

        f_n.append( lambda x_n, m = m, b = b: m * x_n + b )

        f_xn_1 = f( x )
        x = x - learning_rate * f_d( x )
        f_xn = f( x)

        delta_f = abs( f_xn - f_xn_1 )

    print( f"Minima found at x = {x}" )

    return f_n

def f( x: float ) -> float:
    return ( x - 2 ) ** 2

def f_d( x: float ) -> float:
    return  2 * ( x - 2 )

plottable_funcs = gradient_descent_1d_func_gen( f = f, f_d = f_d, x_0 = -2 )
labels = [f"Step {l - 1}" for l, _ in enumerate(plottable_funcs) ]
labels[0] = "f(x)"

plot.plot_univariate_functions( funcs = plottable_funcs, labels = labels, x_range = (-10, 10) ).show()

As shown, the algorithm above computed $x = 1.985$, near to the analytical value found at $x = 2$. A smaller $\epsilon$ (and potentially smaller $\alpha$ to avoid overshooting), would yield an even more accurate result. Using the example above of a function for which finding the analytical solution is not practical, $f(x) = x^2 + e^{-x}$, gradient descent works just as well.

In [ ]:
def f( x: float ) -> float:
    return x ** 2 + np.exp( -x )

def f_d( x: float ) -> float:
    return  2 * x - np.exp( -x )

plottable_funcs = gradient_descent_1d_func_gen( f = f, f_d = f_d, x_0 = -3, learning_rate = 0.2 )
labels = [f"Step {l - 1}" for l, _ in enumerate(plottable_funcs) ]
labels[0] = "f(x)"

plot.plot_univariate_functions( funcs = plottable_funcs, labels = labels, x_range = (-5, 5) ).show()

Unfornately, the last example shows the sensitivity of the algorithm to $\alpha$. Gradient descent can be seen overshooting the solution in step 1, then backtracking to find the minima at $x = 0.364$. To mitigate this, Newton's Method exploits the curvature information contained in $f''(x)$ to set the value of $\alpha$. The equation for Newton's Method is

\begin{align*}
x_{n + 1} = x_n - f''(x)^{-1} f'(x)
\end{align*}

In a simple example, such as $f(x) = x^2$, $f''(x) = 2$, meaning that the ideal learning rate for a simple parabola is $2^{-1} = 0.5$. A plot with this higher learning rate is shown below.

In [ ]:
def newtons_method_1d_func_gen( f: Callable[[float], float], f_d: Callable[[float], float], f_dd: Callable[[float], float], x_0: float, epsilon: float = 0.001 ) -> list[Callable[[float], float]]:
    
    delta_f: float = epsilon * 10
    x: float = x_0

    f_n: list[Callable[[float], float]] = [f]
    
    while( delta_f > epsilon ):
        m = f_d( x )
        b = f( x ) - m * x

        f_n.append( lambda x_n, m = m, b = b: m * x_n + b )

        f_xn_1 = f( x )
        x = x - f_d( x ) / f_dd( x )
        f_xn = f( x)

        delta_f = abs( f_xn - f_xn_1 )

    print( f"Minima found at x = {x}" )

    return f_n

def f( x: float ) -> float:
    return ( x - 2 ) ** 2

def f_d( x: float ) -> float:
    return  2 * ( x - 2 )

def f_dd( x: float ) -> float:
    return 2

plottable_funcs = newtons_method_1d_func_gen( f = f, f_d = f_d, f_dd = f_dd, x_0 = -2 )
labels = [f"Step {l - 1}" for l, _ in enumerate(plottable_funcs) ]
labels[0] = "f(x)"

plot.plot_univariate_functions( funcs = plottable_funcs, labels = labels, x_range = (-10, 10) ).show()

Compared to our guess of $\alpha = 0.2$ using gradient descent, Newton's Method solved for the minima perfectly in 1 iteration. Shown below on $f(x) = x^2 + e^{-x}$, Newton's Method solved for the minima in 3 steps (rather than 7, above), and converged without overshooting.

In [ ]:
def f( x: float ) -> float:
    return x ** 2 + np.exp( -x )

def f_d( x: float ) -> float:
    return  2 * x - np.exp( -x )

def f_dd( x: float ) -> float:
    return 2 + np.exp( -x )

plottable_funcs = newtons_method_1d_func_gen( f = f, f_d = f_d, f_dd = f_dd, x_0 = -2 )
labels = [f"Step {l - 1}" for l, _ in enumerate(plottable_funcs) ]
labels[0] = "f(x)"

plot.plot_univariate_functions( funcs = plottable_funcs, labels = labels, x_range = (-5, 5) ).show()

Extending this to a multivariate problem, the code below shows Newton's method for a classic "bowl" problem in 2 dimensions, centered at $( 2, -5 )$. The equation becomes $f(x_1, x_2) = ( x_1 - 2 )^2 + ( x_2 + 5 )^2 = x_1^2 - 4 x_1 + 4 + x_2^2 + 10 x_2 + 25 = x_1^2 + x_2^2 - 4 x_1 + 10 x_2 + 4 + 25$. In matrix form, the equation is:

\begin{align*}
f(x_1, x_2) = \begin{bmatrix} x_1 & x_2 \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \end{bmatrix} + \begin{bmatrix} -4 & 10 \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \end{bmatrix} + \begin{bmatrix} 4 & 25 \end{bmatrix} \begin{bmatrix} 1 \\ 1 \end{bmatrix}
\end{align*}

and we are now solving the problem $\vec{x}_{n + 1} = \vec{x}_n - \nabla^2 f(\vec{x})^{-1} \nabla f(\vec{x}) = \vec{x}_{n + 1} = \vec{x}_n - H^{-1} J$.

Computing the Jacobian (first derivative), which should have the dimensions of $outputs \times inputs$, yields:

\begin{align*}
J = \begin{bmatrix} \frac{\delta f}{\delta x_1} & \frac{\delta f}{\delta x_2} \end{bmatrix} = \begin{bmatrix} 2 x_1 - 4 & 2 x_2 + 10 \end{bmatrix}
\end{align*}

Finally, computing the Hessian yields:

\begin{align*}
H = \begin{bmatrix} \frac{\delta^2 f}{\delta x_1} & \frac{\delta^2 f}{\delta x_1 x_2} \\ \frac{\delta^2 f}{\delta x_2 x_1} & \frac{\delta^2 f}{\delta x_2} \end{bmatrix} = \begin{bmatrix} 2 & 0 \\ 0 & 2 \end{bmatrix}
\end{align*}

The above equation is a positive definite - i.e. the Eigenvalues are positive and non-zero. This means that the function defines a "bowl"-shape and is solvable as a minimization function. Other options include:

**Positive definite:** All Eigenvalues are positive. Each dimension converges to a minima.

**Negative definite:** All Eigenvalues are negative. Each dimension converges to a maxima.

**Indefinite:** Some Eigenvalues are positive and some are negative. In the 2D case, this defines a saddle.

**Positive semi-definite:** All Eigenvalues are either positive or zero, making the matrix rank-deficient and non-invertable.

**Negative semi-definite:** All Eigenvalues are either negative or zero, making the matrix rank-deficient and non-invertable.


In [ ]:
def newtons_method_2d_func_gen( f: Callable[[float, float], float], f_d: Callable[[float, float], np.ndarray], f_dd: Callable[[float, float], np.ndarray], x_0: np.ndarray, epsilon: float = 0.001, max_iterations: int = 10 ) -> list[Callable[[float, float], float]]:
    
    delta_f: float = epsilon * 10
    iterations: int = max_iterations
    x: np.ndarray = x_0

    f_n: list[Callable[[float, float], float]] = [f]
    
    while( delta_f > epsilon and iterations > 0 ):
        grad = f_d( x[0], x[1] )
        c = f( x[0], x[1] ) - ( grad[0] * x[0] + grad[1] * x[1] ) 

        f_n.append( lambda x_n1, x_n2, a = grad[0], b = grad[1], c = c: float( a * x_n1 + b * x_n2 + c ) )

        f_xn_1 = f( x[0], x[1] )
        x = x - np.linalg.inv( f_dd( x[0], x[1] ) ) @ f_d( x[0], x[1] )
        f_xn = f( x[0], x[1] )

        delta_f = abs( f_xn - f_xn_1 )
        iterations -= 1

    print( f"Critical point found at x = {x}" )

    return f_n

def f( x1: float, x2: float ) -> float:
    return ( x1 - 2 ) ** 2 + ( x2 + 5 ) ** 2

def f_d( x1: float, x2: float ) -> np.ndarray:
    return np.array( [2 * x1 - 4, 2 * x2 + 10] )

def f_dd( x1: float, x2: float ) -> np.ndarray:
    return np.array( [[2, 0], [0, 2]] )

plottable_funcs = newtons_method_2d_func_gen( f = f, f_d = f_d, f_dd = f_dd, x_0 = np.array( [10, 10] ) )
labels = [f"Step {l - 1}" for l, _ in enumerate(plottable_funcs) ]
labels[0] = "f(x)"

plot.plot_multivariate_functions( funcs = plottable_funcs, labels = labels, x_range = (-10, 10), title = "Positive Definite Hessian" ).show()

Below are examples of cost surfaces for each of the non-positive definite Hessian that will lead to instability when using Newton's Method.

**Negative Definite**

\begin{align*}
f(x_1, x_2) = -x_1^2 - x_2^2
\end{align*}

\begin{align*}
J = \begin{bmatrix} \frac{\delta f}{\delta x_1} & \frac{\delta f}{\delta x_2} \end{bmatrix} = \begin{bmatrix} -2x_1 & -2x_2 \end{bmatrix}
\end{align*}

\begin{align*}
H = \begin{bmatrix} \frac{\delta^2 f}{\delta x_1} & \frac{\delta^2 f}{\delta x_1 x_2} \\ \frac{\delta^2 f}{\delta x_2 x_1} & \frac{\delta^2 f}{\delta x_2} \end{bmatrix} = \begin{bmatrix} -2 & 0 \\ 0 & -2 \end{bmatrix}
\end{align*}

This example shows how Newton's method easily adapts to find the only critical point available - a local maxima. This works since the Hessian practically negates the learning rate-term, causing the algorithm to climb rather than descend. This is important for problems where the cost surface is "bumpy" (multiple local critical points) and the cost must be minimized (as in NDT). The solution to this to dampen the Hessian term via $(H + \lambda I)^{-1}$ which, when give an appropriately large $\lambda$, slides the Eigenvalues until a positive definite is produced. This "dampens" the algorithm by decreasing the learning rate. The Levenberg-Marquardt algorithm does this dynamically, where the algorithm begins in a highly-damped phase (closer to gradient descent) before slowly morphing into pure Newton's Method by decreasing $\lambda \rightarrow 0$.

In [ ]:
def f( x1: float, x2: float ) -> float:
    return -x1 ** 2 - x2 ** 2

def f_d( x1: float, x2: float ) -> np.ndarray:
    return np.array( [-2 * x1, -2 * x2] )

def f_dd( x1: float, x2: float ) -> np.ndarray:
    return np.array( [[-2, 0], [0, -2]] )

plottable_funcs = newtons_method_2d_func_gen( f = f, f_d = f_d, f_dd = f_dd, x_0 = np.array( [10, 10] ) )
labels = [f"Step {l - 1}" for l, _ in enumerate(plottable_funcs) ]
labels[0] = "f(x)"

plot.plot_multivariate_functions( funcs = plottable_funcs, labels = labels, x_range = (-10, 10), title = "Negative Definite Hessian" ).show()

**Levenberg Marquardt**

In the example $f(x_1, x_2) = x_1^3 - 3x + x_2^2 - 3x$, if the initial guess were near $(0, 0)$ there are four critical points that Newton's Method could converge to - a maxima, minima, and two saddles. Using pure Newton's Method, the initialization is critical to finding the local minima at $(1, 1)$. Given:

\begin{align*}
J = \begin{bmatrix} \frac{\delta f}{\delta x_1} & \frac{\delta f}{\delta x_2} \end{bmatrix} = \begin{bmatrix} 3x_1^2 - 3 & 3x_2^2 - 3 \end{bmatrix}
\end{align*}

\begin{align*}
H = \begin{bmatrix} \frac{\delta^2 f}{\delta x_1} & \frac{\delta^2 f}{\delta x_1 x_2} \\ \frac{\delta^2 f}{\delta x_2 x_1} & \frac{\delta^2 f}{\delta x_2} \end{bmatrix} = \begin{bmatrix} 6x_1 & 0 \\ 0 & 6x_2 \end{bmatrix}
\end{align*}

we can see that any initialization where $(x_1, x_2)$ are not both positive will result in convergence onto either a saddle point or the local maxima at $(-1, -1)$. Though extending $(x_1, x_2)$ beyond $(-1, -1)$ will result in divergence down the slope in the negative direction, the area for which convergence on the local minima can be guaranteed can be extended using the Levenberg-Marquardt algorithm. If we initialize with $\lambda = 0.001$ in the slightly negative region, the algorithm will begin climbing the hill toward the local maxima. To punish the algorithm for attempting to move up the hill by continually increasing $\lambda$ by a factor of $10$, the algorithm will eventually (once $\lambda = 1$, in this case) converge on the local minima at $(1, 1)$. This works up until intializiation reaches the local maxima at $(-1, -1)$, where the convergence becomes unstable via the "ball on the hill" problem. 

In [ ]:
def f( x1: float, x2: float ) -> float:
    return x1 ** 3 - 3 * x1 + x2 ** 3 - 3 * x2

plot.plot_multivariate_functions( funcs = [f], labels = ["f(x)"], x_range = (-10, 10), title = "Cubic Example" ).show()

In [ ]:
def f( x1: float, x2: float ) -> float:
    return x1 ** 3 - 3 * x1 + x2 ** 3 - 3 * x2

def f_d( x1: float, x2: float ) -> np.ndarray:
    return np.array( [3 * x1 ** 2 - 3, 3 * x2 ** 2 - 3] )

def f_dd( x1: float, x2: float ) -> np.ndarray:
    return np.array( [[6 * x1, 0], [0, 6 * x2]] )

plottable_funcs = newtons_method_2d_func_gen( f = f, f_d = f_d, f_dd = f_dd, x_0 = np.array( [-0.5, -0.5] ) )
labels = [f"Step {l - 1}" for l, _ in enumerate(plottable_funcs) ]
labels[0] = "f(x)"

plot.plot_multivariate_functions( funcs = plottable_funcs, labels = labels, x_range = (-10, 10), title = "Pure Newton's Method with a slight-negative initialization" ).show()

In [ ]:
def levenberg_marquardt_2d_func_gen( f: Callable[[float, float], float], f_d: Callable[[float, float], np.ndarray], f_dd: Callable[[float, float], np.ndarray], x_0: np.ndarray, epsilon: float = 0.001, max_iterations: int = 10 ):
    
    delta_f: float = epsilon * 10
    iterations: int = max_iterations
    x: np.ndarray = x_0
    lbda = 0.001

    f_n: list[Callable[[float, float], float]] = [f]
    
    while( delta_f > epsilon and iterations > 0 ):
        grad = f_d( x[0], x[1] )
        c = f( x[0], x[1] ) - ( grad[0] * x[0] + grad[1] * x[1] ) 

        f_xn_1 = f( x[0], x[1] )
        x_new = x - ( np.linalg.inv( f_dd( x[0], x[1] ) ) + lbda * np.eye( 2 ) ) @ f_d( x[0], x[1] )
        f_xn = f( x_new[0], x_new[1] )

        if( f_xn_1 < f_xn ):
            lbda *= 10
            print( f"Step {max_iterations - iterations} / {max_iterations}:  Score increased - increasing lambda to {lbda} to reverse direction." )

        else:
            lbda /= 10

            f_n.append( lambda x_n1, x_n2, a = grad[0], b = grad[1], c = c: float( a * x_n1 + b * x_n2 + c ) )

            x = x_new
            delta_f = abs( f_xn - f_xn_1 )
            iterations -= 1
            print( f"Step {max_iterations - iterations} / {max_iterations}:  Score decreased - decreasing lambda to {lbda} to speed up progress toward the minima." )

    print( f"Critical point found at x = {x}" )

    return f_n

def f( x1: float, x2: float ) -> float:
    return x1 ** 3 - 3 * x1 + x2 ** 3 - 3 * x2

def f_d( x1: float, x2: float ) -> np.ndarray:
    return np.array( [3 * x1 ** 2 - 3, 3 * x2 ** 2 - 3] )

def f_dd( x1: float, x2: float ) -> np.ndarray:
    return np.array( [[6 * x1, 0], [0, 6 * x2]] )

plottable_funcs = levenberg_marquardt_2d_func_gen( f = f, f_d = f_d, f_dd = f_dd, x_0 = np.array( [-0.5, -0.5] ) )
labels = [f"Step {l - 1}" for l, _ in enumerate(plottable_funcs) ]
labels[0] = "f(x)"

plot.plot_multivariate_functions( funcs = plottable_funcs, labels = labels, x_range = (-10, 10), title = "Levenberg-Marquardt with a slight-negative initialization" ).show()

**Indefinite**

\begin{align*}
f(x_1, x_2) = x_1^2 - x_2^2
\end{align*}

\begin{align*}
J = \begin{bmatrix} \frac{\delta f}{\delta x_1} & \frac{\delta f}{\delta x_2} \end{bmatrix} = \begin{bmatrix} 2x_1 & -2x_2 \end{bmatrix}
\end{align*}

\begin{align*}
H = \begin{bmatrix} \frac{\delta^2 f}{\delta x_1} & \frac{\delta^2 f}{\delta x_1 x_2} \\ \frac{\delta^2 f}{\delta x_2 x_1} & \frac{\delta^2 f}{\delta x_2} \end{bmatrix} = \begin{bmatrix} 2 & 0 \\ 0 & -2 \end{bmatrix}
\end{align*}

In this example, there is a critical point at $(0, 0)$ and Newton's Method converged to that critical point, but it is neither a maxima or minima - it is a saddle point. In this case, the only way to identify the mistake is to check the Eigenvalues.

In [ ]:
def f( x1: float, x2: float ) -> float:
    return x1 ** 2 - x2 ** 2

def f_d( x1: float, x2: float ) -> np.ndarray:
    return np.array( [2 * x1, -2 * x2] )

def f_dd( x1: float, x2: float ) -> np.ndarray:
    return np.array( [[2, 0], [0, -2]] )

plottable_funcs = newtons_method_2d_func_gen( f = f, f_d = f_d, f_dd = f_dd, x_0 = np.array( [5, 10] ) )
labels = [f"Step {l - 1}" for l, _ in enumerate(plottable_funcs) ]
labels[0] = "f(x)"

plot.plot_multivariate_functions( funcs = plottable_funcs, labels = labels, x_range = (-10, 10), title = "Indefinite Hessian" ).show()

In [ ]:
# initial_R = mat.get_dcm( 0, 0, 0 )
# initial_t = np.array( [0, 0, 0] ).reshape( (3, 1) )

# se3 = np.eye(4)

# # se3[:3, :3] = initial_R
# # se3[:3, 3:] = initial_t

# p = Parameters( se3 = se3 )

# if( mesh.mesh is not None ):  

#     # Intialize reference point cloud
#     ref_pc = ReferencePointCloud( y = np.asarray( mesh.mesh.vertices ) )
#     pc_list = ref_pc.get_pc_list()

#     # Initialize target point cloud
#     obs, lbls, pos, dcm = mesh.create_full_sample_observations( n = 1, p = 300, pad = 300 )

#     obs = obs.squeeze( axis = 0 )
#     obs = ( initial_R @ obs.T + initial_t ).T

#     mesh.display_point_clouds( pc_list + [ obs ], [f"Voxel {i}" for i in range( len( pc_list ) )] + list( lbls ), "Sensed and reference point clouds before alignment" )

#     target_pc = TargetPointCloud( p, ref_pc.get_voxel )

#     print( f"Start Pose: {p.to_string()}" )

#     pts: list[Point] = []
#     for pt in obs:
#         pts.append( Point( pt.reshape( ( 3, 1 ) ), ref_pc.get_voxel( pt.reshape( ( 3, 1 ) ) ) ) )
#         target_pc.add( pts[-1] )

#     opt = Optimization()
#     opt.newtons_method( target_pc, se3 )

#     obs = ( p.get_dcm() @ obs.T + p.get_position() ).T
    
#     mesh.display_point_clouds( pc_list + [ obs ], [f"Voxel {i}" for i in range( len( pc_list ) )] + list( lbls ), "Sensed and reference point clouds after alignment" )


# print( f"End Pose: {p.to_string()}" )